In [16]:
import re
import time
import unicodedata
import requests
from bs4 import BeautifulSoup
import pandas as pd

WEIGH_IN_URL = (
    "https://www.ufc.com/news/official-weigh-results-ufc-fight-night-muhammad-vs-bonfim"
)
MATCHES_CSV = "ufc_fights_data.csv"

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
})

BOUT_LINE_RE = re.compile(
    r"^(?:(?P<weight_class>.+?):\s*)?"
    r"(?P<f1>.+?)\s*\((?P<w1>\d+(?:\.\d+)?)\*?\)\s*vs\s*"
    r"(?P<f2>.+?)\s*\((?P<w2>\d+(?:\.\d+)?)\*?\)\*?\s*$",
    re.I,
)


def normalize_weight_class(raw: str | None) -> str | None:
    if not raw:
        return None
    wc = raw.strip()
    wc = re.sub(r"^(?:Main Event|Co-Main Event)\s*-\s*", "", wc, flags=re.I)
    wc = re.sub(r"^UFC\s+", "", wc, flags=re.I)
    wc = re.sub(r"\s+Championship$", "", wc, flags=re.I)
    wc = re.sub(r"\s+Bout$", "", wc, flags=re.I)
    return wc.strip()


FIGHTER_ALIASES = {
    "dariya zheleznyakova": "daria zhelezniakova",
}


def normalize_fighter(name: str) -> str:
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = name.replace("'", "'").replace("'", "'").replace("`", "'")
    name = re.sub(r"[-–—]", " ", name)
    name = re.sub(r"\s+", " ", name.strip().lower())
    name = re.sub(r"\s+iii$", "", name)
    return FIGHTER_ALIASES.get(name, name)


def normalize_event_name(name: str) -> str:
    """Normalize event titles for matching article h2 tags to ufcstats names."""
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = name.lower().replace("'", "'")
    name = re.sub(r"\.", "", name)
    name = re.sub(r"[-–—]", " ", name)
    name = re.sub(r"\s+vs\.?\s+", " vs ", name)
    name = re.sub(r"\s+", " ", name.strip())
    return name


OFFICIAL_WEIGH_IN_RESULTS_RE = re.compile(
    r"\s*Official\s+Weigh[- ]?In\s+Results:?\s*$",
    re.I,
)


def parse_event_name(title: str) -> str:
    """Normalize event title from an article heading."""
    parts = [p.strip() for p in title.split("|") if p.strip()]
    name = parts[1] if len(parts) > 1 else parts[0]
    name = re.sub(r"^Official Weigh-In Results\s*", "", name, flags=re.I).strip()
    name = OFFICIAL_WEIGH_IN_RESULTS_RE.sub("", name).strip()

    fight_night = re.match(r"UFC Fight Night:\s*(.+)", name, re.I)
    if fight_night:
        return f"UFC FIGHT NIGHT: {fight_night.group(1).strip().upper()}"

    numbered = re.match(r"UFC\s+(\d+):\s*(.+)", name, re.I)
    if numbered:
        return f"UFC {numbered.group(1)}: {numbered.group(2).strip()}"

    return name


def event_bout_from_name(event_name: str) -> str:
    """Main event bout portion from an event title."""
    bout = re.sub(r"^UFC Fight Night:\s*", "", event_name, flags=re.I)
    bout = re.sub(r"^UFC\s+\d+:\s*", "", bout, flags=re.I)
    return bout.strip()


def extract_rematch_number(text: str) -> int | None:
    """Trailing rematch number, e.g. 'Holloway vs Oliveira 2' -> 2."""
    match = re.search(r"\s+(\d+)\s*$", text.strip())
    if match:
        return int(match.group(1))
    return None


def strip_rematch_suffix(text: str) -> str:
    return re.sub(r"\s+\d+\s*$", "", text.strip())


def parse_fighters_from_bout(bout: str) -> tuple[str, str] | None:
    """Parse main-event fighters from a bout title, stripping rematch suffixes."""
    bout = strip_rematch_suffix(bout)
    match = re.match(r"(.+?)\s+vs\.?\s+(.+)", bout, re.I)
    if not match:
        return None
    return match.group(1).strip(), match.group(2).strip()


def extract_rematch_number_from_soup(soup: BeautifulSoup) -> int | None:
    h2_name = extract_event_name_from_h2(soup)
    if h2_name:
        rematch = extract_rematch_number(event_bout_from_name(h2_name))
        if rematch is not None:
            return rematch

    h1 = soup.find("h1")
    if h1:
        for part in h1.get_text(strip=True).split("|"):
            part = part.strip()
            if not re.search(r"UFC", part, re.I):
                continue
            numbered = re.search(r"UFC\s+\d+:\s*(.+)", part, re.I)
            if numbered:
                rematch = extract_rematch_number(numbered.group(1))
                if rematch is not None:
                    return rematch
            fight_night = re.search(r"UFC Fight Night:\s*(.+)", part, re.I)
            if fight_night:
                rematch = extract_rematch_number(fight_night.group(1))
                if rematch is not None:
                    return rematch
    return None


def fighter_pair_key(f1: str, f2: str) -> tuple[str, str]:
    return tuple(sorted([normalize_fighter(f1), normalize_fighter(f2)]))


def fighters_match(name_a: str, name_b: str) -> bool:
    """Flexible fighter name comparison for article vs stats spelling differences."""
    a = normalize_fighter(name_a)
    b = normalize_fighter(name_b)
    if a == b:
        return True
    if a in b or b in a:
        return True
    a_last, b_last = a.split()[-1], b.split()[-1]
    return len(a_last) > 3 and a_last == b_last


def pair_matches_bout(f1: str, f2: str, p1: str, p2: str) -> bool:
    return (fighters_match(f1, p1) and fighters_match(f2, p2)) or (
        fighters_match(f1, p2) and fighters_match(f2, p1)
    )


def extract_event_name_from_h2(soup: BeautifulSoup) -> str | None:
    for h2 in soup.find_all("h2"):
        text = h2.get_text(strip=True)
        if re.search(r"official\s+weigh[- ]?in\s+results", text, re.I) and re.match(
            r"UFC", text, re.I
        ):
            return parse_event_name(text)
    return None


def extract_ufc_number(soup: BeautifulSoup) -> str | None:
    for tag in soup.find_all(["h1", "title"]):
        text = tag.get_text(strip=True)
        if not re.search(r"weigh", text, re.I):
            continue

        match = re.search(r"UFC\s+(\d+)\s+WEIGH", text, re.I)
        if match:
            return match.group(1)

        match = re.search(r"UFC\s+(\d+)\s*:", text, re.I)
        if match:
            return match.group(1)

        match = re.search(r"UFC\s+(\d+)\b", text, re.I)
        if match:
            return match.group(1)
    return None


def extract_bout_lines(soup: BeautifulSoup) -> list[dict]:
    rows = []
    seen_lines: set[str] = set()
    for text_node in soup.find_all(string=BOUT_LINE_RE):
        line = text_node.strip()
        if line in seen_lines:
            continue
        seen_lines.add(line)

        match = BOUT_LINE_RE.search(line)
        if not match:
            continue

        rows.append(
            {
                "line": line,
                "weight_class": normalize_weight_class(match.group("weight_class")),
                "f1": match.group("f1").strip(),
                "w1": float(match.group("w1")),
                "f2": match.group("f2").strip(),
                "w2": float(match.group("w2")),
            }
        )
    return rows


def extract_main_event_pair(
    soup: BeautifulSoup, bout_lines: list[dict]
) -> tuple[str, str] | None:
    """Main event fighters from a 'Main Event' section, else the first bout line."""
    if not bout_lines:
        return None

    page_text = soup.get_text("\n")
    main_match = re.search(
        r"Main Event\s*\n\s*"
        r"(?P<f1>.+?)\s*\(\d+(?:\.\d+)?\)\s*vs\s*"
        r"(?P<f2>.+?)\s*\(\d+(?:\.\d+)?\)",
        page_text,
        re.I,
    )
    if main_match:
        return main_match.group("f1").strip(), main_match.group("f2").strip()

    first = bout_lines[0]
    return first["f1"], first["f2"]


def bout_lines_to_dataframe(bout_lines: list[dict], event_name: str) -> pd.DataFrame:
    rows = []
    for bout in bout_lines:
        rows.append(
            {
                "fighter": bout["f1"],
                "weigh_in": bout["w1"],
                "weight_class": bout["weight_class"],
                "event": event_name,
            }
        )
        rows.append(
            {
                "fighter": bout["f2"],
                "weigh_in": bout["w2"],
                "weight_class": bout["weight_class"],
                "event": event_name,
            }
        )
    return pd.DataFrame(rows)


def build_ufc_number_index(stats_events: dict[str, str]) -> dict[str, str]:
    index: dict[str, str] = {}
    for event_date, event_name in stats_events.items():
        match = re.match(r"UFC\s+(\d+)\s*:", event_name, re.I)
        if match:
            index[match.group(1)] = event_date
    return index


def find_event_dates_by_main_event_pair(
    f1: str,
    f2: str,
    fights_df: pd.DataFrame,
) -> list[str]:
    matches: list[str] = []
    for event_date in fights_df["event_date"].drop_duplicates():
        card = fights_df[fights_df["event_date"] == event_date]
        if card.empty:
            continue
        main = card.iloc[0]
        if pair_matches_bout(f1, f2, main["p1_fighter"], main["p2_fighter"]):
            matches.append(event_date)
    return matches


def disambiguate_event_dates(
    candidates: list[str],
    stats_events: dict[str, str],
    weigh_in_df: pd.DataFrame,
    fights_df: pd.DataFrame,
    ufc_number: str | None = None,
    rematch_number: int | None = None,
) -> str | None:
    if not candidates:
        return None
    if len(candidates) == 1:
        return candidates[0]

    if ufc_number:
        filtered = [
            event_date
            for event_date in candidates
            if re.match(rf"UFC\s+{ufc_number}\s*:", stats_events[event_date], re.I)
        ]
        if len(filtered) == 1:
            return filtered[0]
        if filtered:
            candidates = filtered

    if rematch_number is not None:
        filtered = [
            event_date
            for event_date in candidates
            if extract_rematch_number(event_bout_from_name(stats_events[event_date]))
            == rematch_number
        ]
        if len(filtered) == 1:
            return filtered[0]
        if filtered:
            candidates = filtered

    weigh_fighters = {normalize_fighter(n) for n in weigh_in_df["fighter"]}
    best_date = None
    best_count = 0
    for event_date in candidates:
        card = fights_df[fights_df["event_date"] == event_date]
        event_fighters = set(card["p1_fighter"].map(normalize_fighter)) | set(
            card["p2_fighter"].map(normalize_fighter)
        )
        overlap = len(weigh_fighters & event_fighters)
        if overlap > best_count:
            best_count = overlap
            best_date = event_date
    return best_date


def find_event_by_main_event_pair(
    f1: str,
    f2: str,
    fights_df: pd.DataFrame,
    stats_events: dict[str, str],
    weigh_in_df: pd.DataFrame,
    ufc_number: str | None = None,
    rematch_number: int | None = None,
) -> str | None:
    candidates = find_event_dates_by_main_event_pair(f1, f2, fights_df)
    return disambiguate_event_dates(
        candidates,
        stats_events,
        weigh_in_df,
        fights_df,
        ufc_number=ufc_number,
        rematch_number=rematch_number,
    )


def find_event_by_fighter_overlap(
    weigh_in_df: pd.DataFrame,
    fights_df: pd.DataFrame,
    min_overlap: int = 4,
) -> str | None:
    weigh_fighters = {normalize_fighter(n) for n in weigh_in_df["fighter"]}
    best_date = None
    best_count = 0

    for event_date in fights_df["event_date"].drop_duplicates():
        card = fights_df[fights_df["event_date"] == event_date]
        event_fighters = set(card["p1_fighter"].map(normalize_fighter)) | set(
            card["p2_fighter"].map(normalize_fighter)
        )
        overlap = len(weigh_fighters & event_fighters)
        if overlap > best_count:
            best_count = overlap
            best_date = event_date

    if best_date and best_count >= min_overlap:
        return best_date
    return None


def match_article_to_event(
    soup: BeautifulSoup,
    weigh_in_df: pd.DataFrame,
    fights_df: pd.DataFrame,
    stats_events: dict[str, str],
    event_name_index: dict[str, str],
    ufc_number_index: dict[str, str],
    bout_lines: list[dict],
) -> tuple[str | None, str, str]:
    """Return (event_date, event_name, match_method)."""
    ufc_number = extract_ufc_number(soup)
    rematch_number = extract_rematch_number_from_soup(soup)

    h2_name = extract_event_name_from_h2(soup)
    if h2_name:
        event_date = event_name_index.get(
            normalize_event_name(
                h2_name.replace("UFC FIGHT NIGHT:", "UFC Fight Night:")
            )
        )
        if event_date and rematch_number is not None:
            event_rematch = extract_rematch_number(
                event_bout_from_name(stats_events[event_date])
            )
            if event_rematch != rematch_number:
                event_date = None
        if event_date:
            return event_date, stats_events[event_date], "h2"

        h2_pair = parse_fighters_from_bout(event_bout_from_name(h2_name))
        if h2_pair:
            event_date = find_event_by_main_event_pair(
                h2_pair[0],
                h2_pair[1],
                fights_df,
                stats_events,
                weigh_in_df,
                ufc_number=ufc_number,
                rematch_number=rematch_number,
            )
            if event_date:
                return event_date, stats_events[event_date], "h2"

    if ufc_number:
        event_date = ufc_number_index.get(ufc_number)
        if event_date:
            return event_date, stats_events[event_date], "ufc_number"

    main_pair = extract_main_event_pair(soup, bout_lines)
    if main_pair:
        event_date = find_event_by_main_event_pair(
            main_pair[0],
            main_pair[1],
            fights_df,
            stats_events,
            weigh_in_df,
            ufc_number=ufc_number,
            rematch_number=rematch_number,
        )
        if event_date:
            return event_date, stats_events[event_date], "main_event_pair"

    event_date = find_event_by_fighter_overlap(weigh_in_df, fights_df)
    if event_date:
        return event_date, stats_events[event_date], "fighter_overlap"

    return None, "", ""


def scrape_weigh_in_page(url: str, retries: int = 3) -> tuple[BeautifulSoup, list[dict], pd.DataFrame]:
    """Fetch a weigh-in article and parse bout lines without requiring h2 tags."""
    last_error = None
    for attempt in range(retries):
        try:
            response = SESSION.get(url, timeout=30)
            response.raise_for_status()
            break
        except requests.RequestException as e:
            last_error = e
            if attempt < retries - 1:
                time.sleep(2 * (attempt + 1))
            else:
                raise last_error

    soup = BeautifulSoup(response.text, "html.parser")
    bout_lines = extract_bout_lines(soup)
    if not bout_lines:
        raise ValueError("No weigh-in results found on page")

    placeholder_event = extract_event_name_from_h2(soup) or "Unknown Event"
    weigh_in_df = bout_lines_to_dataframe(bout_lines, placeholder_event)
    return soup, bout_lines, weigh_in_df


def scrape_weigh_ins(url: str, retries: int = 3) -> tuple[str, pd.DataFrame]:
    """Scrape fighter weigh-in weights from a UFC.com official weigh-in article."""
    soup, bout_lines, weigh_in_df = scrape_weigh_in_page(url, retries=retries)
    event_name = extract_event_name_from_h2(soup)
    if not event_name:
        main_pair = extract_main_event_pair(soup, bout_lines)
        if main_pair:
            event_name = f"{main_pair[0]} vs {main_pair[1]}"
        else:
            h1 = soup.find("h1")
            event_name = h1.get_text(strip=True) if h1 else "Unknown Event"

    weigh_in_df = bout_lines_to_dataframe(bout_lines, event_name)
    return event_name, weigh_in_df


def validate_event_weigh_ins(
    weigh_in_df: pd.DataFrame,
    fights_df: pd.DataFrame,
    event_date: str,
) -> tuple[pd.DataFrame, dict]:
    """Cross-reference weigh-ins against a known event date from scrapeMatches."""
    event_fights = fights_df[fights_df["event_date"] == event_date].copy()
    weigh_in_fighters = {normalize_fighter(n) for n in weigh_in_df["fighter"]}
    event_fighters = set(event_fights["p1_fighter"].map(normalize_fighter)) | set(
        event_fights["p2_fighter"].map(normalize_fighter)
    )

    matched = weigh_in_fighters & event_fighters
    missing_from_weigh_in = sorted(event_fighters - weigh_in_fighters)
    extra_on_weigh_in = sorted(weigh_in_fighters - event_fighters)

    validated = validate_weigh_ins(weigh_in_df, event_fights)
    summary = {
        "matched_event_date": event_date,
        "weigh_in_fighters": len(weigh_in_fighters),
        "matched_fighters": len(matched),
        "event_card_fighters": len(event_fighters),
        "all_fighters_match": not missing_from_weigh_in and not extra_on_weigh_in,
        "missing_from_weigh_in": missing_from_weigh_in,
        "extra_on_weigh_in": extra_on_weigh_in,
    }
    return validated, summary


def validate_weigh_ins(
    weigh_in_df: pd.DataFrame, event_fights: pd.DataFrame
) -> pd.DataFrame:
    """Flag fighters that appear on the weigh-in page but not on the matched event card."""
    if event_fights.empty:
        weigh_in_df = weigh_in_df.copy()
        weigh_in_df["on_event_card"] = False
        return weigh_in_df

    event_fighters = set(event_fights["p1_fighter"].map(normalize_fighter)) | set(
        event_fights["p2_fighter"].map(normalize_fighter)
    )

    validated = weigh_in_df.copy()
    validated["on_event_card"] = validated["fighter"].map(
        lambda n: normalize_fighter(n) in event_fighters
    )
    return validated

In [17]:
# Load scrapeMatches output
fights_df = pd.read_csv(MATCHES_CSV)
print(f"Loaded {len(fights_df)} fights from {MATCHES_CSV}")

# Scrape weigh-in page
event_name, weigh_in_df = scrape_weigh_ins(WEIGH_IN_URL)
print(f"\nScraped event: {event_name}")
print(f"Weigh-in records: {len(weigh_in_df)}")

# Cross-reference with scrapeMatches using the known event date
event_weigh_ins, match_summary = validate_event_weigh_ins(
    weigh_in_df, fights_df, "June 06, 2026"
)

print("\n--- Event cross-reference ---")
print(f"Event exists in scrapeMatches: {match_summary['matched_event_date'] is not None}")
print(f"Matched event date: {match_summary['matched_event_date']}")
print(
    f"Fighters matched: {match_summary['matched_fighters']}/{match_summary['weigh_in_fighters']} "
    f"(event card has {match_summary['event_card_fighters']} fighters)"
)
print(f"Full card alignment: {match_summary['all_fighters_match']}")

weigh_in_df = event_weigh_ins

if not match_summary["all_fighters_match"]:
    if match_summary["missing_from_weigh_in"]:
        print(f"On card but missing from weigh-in page: {match_summary['missing_from_weigh_in']}")
    if match_summary["extra_on_weigh_in"]:
        print(f"On weigh-in page but not on card: {match_summary['extra_on_weigh_in']}")

weigh_in_df

Loaded 8705 fights from ufc_fights_data.csv

Scraped event: UFC FIGHT NIGHT: MUHAMMAD VS BONFIM
Weigh-in records: 24

--- Event cross-reference ---
Event exists in scrapeMatches: True
Matched event date: June 06, 2026
Fighters matched: 24/24 (event card has 24 fighters)
Full card alignment: True


,fighter,weigh_in,weight_class,event,on_event_card
0,Belal Muhammad,170.5,Welterweight,UFC FIGHT NIGHT: MUHAMMAD VS BONFIM,True
1,Gabriel Bonfim,170.5,Welterweight,UFC FIGHT NIGHT: MUHAMMAD VS BONFIM,True
2,Brendan Allen,185.5,Middleweight,UFC FIGHT NIGHT: MUHAMMAD VS BONFIM,True
3,Edmen Shahbazyan,186.0,Middleweight,UFC FIGHT NIGHT: MUHAMMAD VS BONFIM,True
4,Fares Ziam,156.0,Lightweight,UFC FIGHT NIGHT: MUHAMMAD VS BONFIM,True
5,Tom Nolan,155.0,Lightweight,UFC FIGHT NIGHT: MUHAMMAD VS BONFIM,True
6,Bryce Mitchell,135.5,Bantamweight,UFC FIGHT NIGHT: MUHAMMAD VS BONFIM,True
7,Santiago Luna,136.0,Bantamweight,UFC FIGHT NIGHT: MUHAMMAD VS BONFIM,True
8,Iwo Baraniewski,206.0,Light Heavyweight,UFC FIGHT NIGHT: MUHAMMAD VS BONFIM,True
9,Junior Tafa,206.0,Light Heavyweight,UFC FIGHT NIGHT: MUHAMMAD VS BONFIM,True


In [18]:
import hashlib
from urllib.parse import urljoin, urlparse

WEIGH_IN_SEARCH_BASE = (
    "https://www.ufc.com/search?query=weigh%20in%20&type=news&page="
)


def fetch_all_weigh_in_article_urls() -> list[str]:
    """Collect every weigh-in news article from paginated UFC.com search results."""
    urls: list[str] = []
    seen: set[str] = set()
    page = 0

    while True:
        search_url = f"{WEIGH_IN_SEARCH_BASE}{page}"
        response = SESSION.get(search_url, timeout=30)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        page_links: list[str] = []
        for anchor in soup.find_all("a", href=True):
            href = anchor["href"]
            if href.startswith("https://www.ufc.com"):
                href = href.replace("https://www.ufc.com", "", 1)
            if not href.startswith("/news/"):
                continue
            if "weigh" not in href.lower():
                continue
            if href in seen:
                continue
            seen.add(href)
            page_links.append(urljoin("https://www.ufc.com", href))

        if not page_links:
            break

        urls.extend(page_links)
        page += 1
        time.sleep(0.2)

    return urls


def fetch_ufcstats_events() -> dict[str, str]:
    """Return {event_date: event_name} for all completed events on ufcstats."""

    def fetch_page(url):
        response = SESSION.get(url)
        if "Checking your browser" not in response.text:
            return response
        match = re.search(r'var nonce="([^"]+)"', response.text)
        if not match:
            return response
        nonce = match.group(1)
        n = 0
        while not hashlib.sha256(f"{nonce}:{n}".encode()).hexdigest().startswith("00"):
            n += 1
        base = f"{urlparse(url).scheme}://{urlparse(url).netloc}"
        SESSION.post(
            f"{base}/__c",
            data={"nonce": nonce, "n": n},
            headers={"Content-Type": "application/x-www-form-urlencoded"},
        )
        return SESSION.get(url)

    response = fetch_page("http://ufcstats.com/statistics/events/completed?page=all")
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")

    events = {}
    for row in soup.select("tr.b-statistics__table-row"):
        link = row.select_one("a.b-link.b-link_style_black")
        if not link:
            continue
        date_el = row.select_one("td")
        date_text = date_el.get_text(" ", strip=True) if date_el else ""
        date_match = re.search(r"([A-Za-z]+ \d{1,2}, \d{4})", date_text)
        if date_match:
            events[date_match.group(1)] = link.text.strip()
    return events


def build_event_name_index(stats_events: dict[str, str]) -> dict[str, str]:
    """Map normalized ufcstats event names back to event dates."""
    index: dict[str, str] = {}
    for event_date, event_name in stats_events.items():
        index[normalize_event_name(event_name)] = event_date
    return index


if "fights_df" not in dir():
    fights_df = pd.read_csv(MATCHES_CSV)

stats_events = fetch_ufcstats_events()
event_name_index = build_event_name_index(stats_events)
ufc_number_index = build_ufc_number_index(stats_events)
all_event_dates = fights_df["event_date"].drop_duplicates().tolist()

print("Collecting weigh-in articles from UFC.com search...")
article_urls = fetch_all_weigh_in_article_urls()
print(f"Found {len(article_urls)} weigh-in news articles")

all_weigh_ins = []
scrape_log = []
best_by_event: dict[str, dict] = {}
total_articles = len(article_urls)

for i, url in enumerate(article_urls, 1):
    if i % 25 == 0 or i == 1:
        print(f"Processing article {i}/{total_articles}...")

    time.sleep(0.15)

    try:
        soup, bout_lines, event_weigh_ins = scrape_weigh_in_page(url)
        event_date, event_name, match_method = match_article_to_event(
            soup,
            event_weigh_ins,
            fights_df,
            stats_events,
            event_name_index,
            ufc_number_index,
            bout_lines,
        )

        if event_date is None:
            scrape_log.append(
                {
                    "url": url,
                    "article_event": None,
                    "event_date": None,
                    "event_name": None,
                    "match_method": None,
                    "status": "skipped: no matching event in matches csv",
                }
            )
            continue

        if event_date not in all_event_dates:
            scrape_log.append(
                {
                    "url": url,
                    "article_event": event_name,
                    "event_date": event_date,
                    "event_name": stats_events.get(event_date),
                    "match_method": match_method,
                    "status": "skipped: event not in matches csv",
                }
            )
            continue

        event_weigh_ins = bout_lines_to_dataframe(bout_lines, event_name)
        event_weigh_ins, summary = validate_event_weigh_ins(
            event_weigh_ins, fights_df, event_date
        )

        if summary["matched_fighters"] < 2:
            scrape_log.append(
                {
                    "url": url,
                    "article_event": event_name,
                    "event_date": event_date,
                    "event_name": stats_events[event_date],
                    "match_method": match_method,
                    "status": (
                        "skipped: too few fighters matched "
                        f"({summary['matched_fighters']}/{summary['event_card_fighters']})"
                    ),
                }
            )
            continue

        event_weigh_ins["event_date"] = event_date
        event_weigh_ins["weigh_in_url"] = url

        entry = {
            "url": url,
            "article_event": event_name,
            "event_date": event_date,
            "event_name": stats_events[event_date],
            "scraped_event": event_name,
            "match_method": match_method,
            "matched_fighters": summary["matched_fighters"],
            "event_card_fighters": summary["event_card_fighters"],
            "all_fighters_match": summary["all_fighters_match"],
            "missing_from_weigh_in": ", ".join(summary["missing_from_weigh_in"]),
            "event_weigh_ins": event_weigh_ins,
            "status": "ok",
        }

        existing = best_by_event.get(event_date)
        if existing is None or entry["matched_fighters"] > existing["matched_fighters"]:
            best_by_event[event_date] = entry

    except ValueError as e:
        scrape_log.append(
            {
                "url": url,
                "article_event": None,
                "event_date": None,
                "event_name": None,
                "match_method": None,
                "status": f"skipped: {e}",
            }
        )
    except Exception as e:
        scrape_log.append(
            {
                "url": url,
                "article_event": None,
                "event_date": None,
                "event_name": None,
                "match_method": None,
                "status": f"error: {e}",
            }
        )

for event_date, entry in best_by_event.items():
    event_weigh_ins = entry.pop("event_weigh_ins")
    all_weigh_ins.append(event_weigh_ins)
    scrape_log.append(entry)
    print(
        f"{event_date} | {entry['event_name']} - ok "
        f"({len(event_weigh_ins)} records, "
        f"{entry['matched_fighters']}/{entry['event_card_fighters']} matched)"
    )

weigh_ins_df = pd.concat(all_weigh_ins, ignore_index=True) if all_weigh_ins else pd.DataFrame()
WEIGH_INS_CSV = "ufc_weigh_ins.csv"
weigh_ins_df.to_csv(WEIGH_INS_CSV, index=False)
print(f"Saved {len(weigh_ins_df):,} weigh-in records to {WEIGH_INS_CSV}")

scrape_log_df = pd.DataFrame(scrape_log)

matched_events = set(best_by_event)
missing_events = [
    {"event_date": d, "event_name": stats_events[d]}
    for d in all_event_dates
    if d in stats_events and d not in matched_events
]

ok_count = len(best_by_event)
print(f"\n{'=' * 60}")
print(f"Matched {ok_count}/{len(all_event_dates)} events from {len(article_urls)} articles")
print(f"Total weigh-in records: {len(weigh_ins_df)}")
print(f"Events still missing weigh-in articles: {len(missing_events)}")
if not scrape_log_df.empty:
    ok_rows = scrape_log_df[scrape_log_df["status"] == "ok"]
    if not ok_rows.empty and "match_method" in ok_rows.columns:
        print("\nMatch methods:")
        print(ok_rows["match_method"].value_counts().to_string())
if missing_events[:10]:
    print("First missing events:")
    for item in missing_events[:10]:
        print(f"  {item['event_date']} | {item['event_name']}")

scrape_log_df


Found 761 weigh-in news articles
Processing article 1/761...
Processing article 25/761...
Processing article 50/761...
Processing article 75/761...
Processing article 100/761...
Processing article 125/761...
Processing article 150/761...
Processing article 175/761...
Processing article 200/761...
Processing article 225/761...
Processing article 250/761...
Processing article 275/761...
Processing article 300/761...
Processing article 325/761...
Processing article 350/761...
Processing article 375/761...
Processing article 400/761...
Processing article 425/761...
Processing article 450/761...
Processing article 475/761...
Processing article 500/761...
Processing article 525/761...
Processing article 550/761...
Processing article 575/761...
Processing article 600/761...
Processing article 625/761...
Processing article 650/761...
Processing article 675/761...
Processing article 700/761...
Processing article 725/761...
Processing article 750/761...
June 06, 2026 | UFC Fight Night: Muhammad 

,url,article_event,event_date,event_name,match_method,status,scraped_event,matched_fighters,event_card_fighters,all_fighters_match,missing_from_weigh_in
0,https://www.ufc.com/news/official-weigh-result...,NaN,NaN,NaN,NaN,skipped: no matching event in matches csv,NaN,NaN,NaN,NaN,NaN
1,https://www.ufc.com/news/official-weigh-in-res...,NaN,NaN,NaN,NaN,skipped: no matching event in matches csv,NaN,NaN,NaN,NaN,NaN
2,https://www.ufc.com/news/official-weigh-result...,NaN,NaN,NaN,NaN,skipped: No weigh-in results found on page,NaN,NaN,NaN,NaN,NaN
3,https://www.ufc.com/news/official-weigh-result...,NaN,NaN,NaN,NaN,skipped: no matching event in matches csv,NaN,NaN,NaN,NaN,NaN
4,https://www.ufc.com/news/official-weigh-result...,NaN,NaN,NaN,NaN,skipped: no matching event in matches csv,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
736,https://www.ufc.com/news/ufc-105-weigh-results,UFC 105: Couture vs Vera,"November 14, 2009",UFC 105: Couture vs Vera,ufc_number,ok,UFC 105: Couture vs Vera,21.0,22.0,False,rolando delgado
737,https://www.ufc.com/news/official-ufc-106-weig...,UFC 106: Ortiz vs Griffin 2,"November 21, 2009",UFC 106: Ortiz vs Griffin 2,ufc_number,ok,UFC 106: Ortiz vs Griffin 2,18.0,20.0,False,"luiz cane, rogerio nogueira"
738,https://www.ufc.com/news/official-tuf-9-finale...,The Ultimate Fighter: United States vs. United...,"June 20, 2009",The Ultimate Fighter: United States vs. United...,main_event_pair,ok,The Ultimate Fighter: United States vs. United...,20.0,20.0,True,
739,https://www.ufc.com/news/ufc-104-weigh-results,UFC 104: Machida vs Shogun,"October 24, 2009",UFC 104: Machida vs Shogun,ufc_number,ok,UFC 104: Machida vs Shogun,15.0,22.0,False,"anthony johnson, antoni hardonk, eric schafer,..."


In [19]:
AUG_CSV = "ufc_aug.csv"
WEIGH_INS_CSV = "ufc_weigh_ins.csv"
OUTPUT_CSV = "ufc_aug_weigh_in.csv"


def parse_event_date(date_str) -> pd.Timestamp | None:
    """Normalize event dates across ufc_aug (ISO) and weigh_ins_df (Month DD, YYYY)."""
    parsed = pd.to_datetime(date_str, errors="coerce")
    return None if pd.isna(parsed) else parsed


def build_weigh_in_index(weigh_ins_df: pd.DataFrame) -> dict:
    """Index validated weigh-in records by event date."""
    index: dict = {}
    if weigh_ins_df.empty:
        return index

    df = weigh_ins_df.copy()
    if "on_event_card" in df.columns:
        df = df[df["on_event_card"].fillna(True)]

    for _, row in df.iterrows():
        event_dt = parse_event_date(row["event_date"])
        if event_dt is None:
            continue
        event_key = event_dt.date()
        index.setdefault(event_key, []).append(
            (normalize_fighter(row["fighter"]), row["fighter"], float(row["weigh_in"]))
        )
    return index


def lookup_weigh_in(
    weigh_in_index: dict,
    event_date,
    fighter_name: str,
) -> float | None:
    event_dt = parse_event_date(event_date)
    if event_dt is None:
        return None

    candidates = weigh_in_index.get(event_dt.date(), [])
    for _, original_name, weight in candidates:
        if fighters_match(fighter_name, original_name):
            return weight
    return None


def attach_weigh_in_columns(fights_df: pd.DataFrame, weigh_in_index: dict) -> pd.DataFrame:
    out = fights_df.copy()
    out["p1_weigh_in"] = out.apply(
        lambda row: lookup_weigh_in(weigh_in_index, row["event_date"], row["p1_fighter"]),
        axis=1,
    )
    out["p2_weigh_in"] = out.apply(
        lambda row: lookup_weigh_in(weigh_in_index, row["event_date"], row["p2_fighter"]),
        axis=1,
    )
    out["weigh_in_diff"] = out["p1_weigh_in"] - out["p2_weigh_in"]
    return out


# Load augmented training data and scraped weigh-ins from CSV
aug_df = pd.read_csv(AUG_CSV)
weigh_ins_df = pd.read_csv(WEIGH_INS_CSV)
if weigh_ins_df.empty:
    raise ValueError(f"{WEIGH_INS_CSV} is empty — run the bulk scrape cell first.")

weigh_in_index = build_weigh_in_index(weigh_ins_df)

# ufc_aug.csv is already augmented; map weigh-ins per row so swapped rows stay consistent
aug_with_weigh_ins = attach_weigh_in_columns(aug_df, weigh_in_index)

matched_rows = aug_with_weigh_ins["p1_weigh_in"].notna() & aug_with_weigh_ins["p2_weigh_in"].notna()
print(f"Loaded {len(aug_df):,} augmented rows from {AUG_CSV}")
print(f"Loaded {len(weigh_ins_df):,} weigh-in records from {WEIGH_INS_CSV}")
print(f"Rows with both weigh-ins mapped: {matched_rows.sum():,} ({matched_rows.mean():.1%})")
print(f"Unique events with weigh-in data: {weigh_ins_df['event_date'].nunique()}")

aug_with_weigh_ins.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {OUTPUT_CSV}")

aug_with_weigh_ins[
    ["p1_fighter", "p2_fighter", "event_date", "p1_weight", "p2_weight", "p1_weigh_in", "p2_weigh_in", "weigh_in_diff"]
].dropna(subset=["p1_weigh_in", "p2_weigh_in"]).head(10)

Loaded 17,508 augmented rows from ufc_aug.csv
Rows with both weigh-ins mapped: 9,530 (54.4%)
Unique events with weigh-in data: 597
Saved ufc_aug_weigh_in.csv


,p1_fighter,p2_fighter,event_date,p1_weight,p2_weight,p1_weigh_in,p2_weigh_in,weigh_in_diff
1798,Jorge Rivera,Martin Kampmann,2008-06-07,185.0,170.0,185.0,186.0,-1.0
1799,Matt Hughes,Thiago Alves,2008-06-07,170.0,170.0,170.0,174.0,-4.0
1800,Marcus Davis,Mike Swick,2008-06-07,170.0,170.0,170.0,170.0,0.0
1801,Nate Marquardt,Thales Leites,2008-06-07,185.0,185.0,185.0,185.0,0.0
1802,Brandon Vera,Fabricio Werdum,2008-06-07,230.0,231.0,228.0,247.0,-19.0
1803,Martin Kampmann,Jorge Rivera,2008-06-07,170.0,185.0,186.0,185.0,1.0
1804,Eddie Sanchez,Antoni Hardonk,2008-06-07,230.0,245.0,244.0,247.0,-3.0
1805,Thiago Tavares,Matt Wiman,2008-06-07,145.0,155.0,154.5,155.0,-0.5
1806,Roan Carneiro,Kevin Burns,2008-06-07,170.0,170.0,171.0,170.0,1.0
1807,Jason Lambert,Luiz Cane,2008-06-07,185.0,185.0,205.0,204.0,1.0


In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.impute import SimpleImputer
import xgboost as xgb

# Import the first CSV file
feature_df = pd.read_csv('ufc_aug.csv')

#prepare data for training

# Drop the columns
columns_to_drop = ['p1_fighter', 'p2_fighter', 'event_date'] #method
feature_df = feature_df.drop(columns=columns_to_drop)
cols_to_drop = [col for col in feature_df.columns if col.startswith('method_')]
feature_df.drop(columns=cols_to_drop, inplace=True)

# Clean all column names
def clean_column_name(col):
    return col.lower().replace(' ', '_').replace('.', '').replace('-', '_')

# Apply to all columns
feature_df.columns = [clean_column_name(col) for col in feature_df.columns]

# Identify all categorical columns
categorical_cols = ['p1_stance', 'p2_stance']

# One-hot encode all categorical variables
feature_df = pd.get_dummies(feature_df, columns=categorical_cols)

feature_df = feature_df.sample(frac=1, random_state=42).reset_index(drop=True)

# encode the referee using frequency
ref_counts = feature_df['referee'].value_counts()
feature_df['referee_freq'] = feature_df['referee'].map(ref_counts)
feature_df.drop(columns=['referee'], inplace=True)

# Target setup
X = feature_df.drop(columns=['winner'])
y = feature_df['winner']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Identify numeric columns for imputation
numeric_cols = X_train.select_dtypes(include=['float64', 'int64']).columns.tolist()

# Impute missing values with median
num_imputer = SimpleImputer(strategy='median')
X_train[numeric_cols] = num_imputer.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = num_imputer.transform(X_test[numeric_cols])

# XGBoost model training with evaluation tracking
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Define evaluation sets for learning curves
evalset = [(X_train, y_train), (X_test, y_test)]

# Fit model with evaluation tracking
xgb_model.fit(X_train, y_train, eval_set=evalset, verbose=False)

# Predictions and evaluation
xgb_pred = xgb_model.predict(X_test)
accuracy = accuracy_score(y_test, xgb_pred)
report = classification_report(y_test, xgb_pred)

print("XGBoost Model Performance:")
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(report)

# Feature importance analysis
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\nFeature Importances:")
print(feature_importance.to_string())

# Save the model
# xgb_model.save_model('xgb_model_good.json')

# Get the underlying booster and save that
xgb_model.get_booster().save_model('xgb_model_good.json')


/var/folders/nc/04zwxn450f3cllk4tzq8_fp40000gn/T/ipykernel_17491/3800749057.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feature_df['referee_freq'] = feature_df['referee'].map(ref_counts)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [00:32:03] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Model Performance:
Accuracy: 0.6759
Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.67      0.67      1744
           1       0.68      0.68      0.68      1758

    accuracy                           0.68      3502
   macro avg       0.68      0.68      0.68      3502
weighted avg       0.68      0.68      0.68      3502


Feature Importances:
                            Feature  Importance
28                        slpm_diff    0.028919
30                        sapm_diff    0.019599
34                       tddef_diff    0.017273
29                      stracc_diff    0.015583
10                       p1_sub_avg    0.013600
53          p2_age_adjusted_str_acc    0.012036
27                         age_diff    0.011594
31                      strdef_diff    0.011532
21                       p2_sub_avg    0.010511
52          p1_age_adjusted_str_acc    0.010366
51             p2_age_adjusted_slpm    0.010056
64      